# DEL PARQUET ANTERIOR AGREGAR ONT_ID Y RX_AVG

In [1]:
import pandas as pd #Importar librería pandas
import os #Importar módulo sistema
import glob #Importar módulo búsqueda archivos
import json #Importar módulo json
from datetime import datetime #Importar formateo de fechas

# Cargar configuración desde JSON
with open('config_etl_silver.json', 'r') as file: #Abrir archivo JSON
    config = json.load(file) #Cargar datos del JSON

extr_anio = config['etl_params']['anio'] #Definir año a procesar desde JSON
extr_mes = config['etl_params']['mes'] #Definir mes a procesar desde JSON

# Construir el formato AAMM (ej. 2026, mes 2 -> 2602)
str_yymm = datetime(extr_anio, extr_mes, 1).strftime('%y%m') #Formatear fecha

RUTA_SILVER_SERVICES = "datalake/silver/service_orders" #Definir ruta origen silver
RUTA_SILVER_RX_AVG = "datalake/silver/rx_avg" #Definir ruta rx_avg silver
RUTA_BRONZE_ZABBIX = "datalake/bronze/zabbix_final" #Definir ruta origen zabbix

if not os.path.exists(RUTA_SILVER_RX_AVG): #Verificar existencia directorio
    os.makedirs(RUTA_SILVER_RX_AVG) #Crear directorio destino

# Archivo Silver generado en el paso anterior (Dinámico con AAMM)
FILE_INPUT_SILVER = f"{RUTA_SILVER_SERVICES}/master_services_silver_{str_yymm}.parquet" #Definir archivo entrada

# Archivos de salida (Nuevo Silver Enriquecido dinámico con AAMM)
FILE_OUTPUT_PARQUET = f"{RUTA_SILVER_RX_AVG}/master_rx+services_silver_{str_yymm}.parquet" #Definir salida parquet
FILE_OUTPUT_CSV = f"{RUTA_SILVER_RX_AVG}/master_rx+services_silver_{str_yymm}.csv" #Definir salida csv

def unir_silver_zabbix(): #Definir función principal
    print(f"Iniciando Enriquecimiento Silver con Zabbix para el periodo {str_yymm}...") #Imprimir inicio proceso

    try: #Iniciar bloque manejo de errores
        print("   Cargando Master Silver...") #Imprimir estado carga
        df_silver = pd.read_parquet(FILE_INPUT_SILVER) #Cargar parquet servicios
        df_silver['CONTRATO'] = df_silver['CONTRATO'].astype(str).str.strip() #Limpiar llave contrato

        print("   Cargando Zabbix (ont_id + rx_avg)...") #Imprimir estado zabbix
        # Listar SOLO los archivos Zabbix que correspondan al mes procesado
        archivos_zabbix = glob.glob(f"{RUTA_BRONZE_ZABBIX}/*_{str_yymm}.parquet") #Listar archivos zabbix filtrados
        
        lista_dfs = [] #Inicializar lista dataframes
        for f in archivos_zabbix: #Iterar archivos
            #Leer ont_id (para conservar) y rx_avg (dato técnico)
            df_temp = pd.read_parquet(f, columns=['ont_id', 'rx_avg']) #Leer columnas especificas
            lista_dfs.append(df_temp) #Agregar a lista

        if not lista_dfs: #Validar lista vacía
            print(f"[ERROR] No se encontraron archivos de Zabbix para el mes {str_yymm}") #Imprimir error
            return #Detener ejecución

        df_zabbix = pd.concat(lista_dfs, ignore_index=True) #Concatenar dataframes
        
        #Crear columna CONTRATO basada en ont_id para el cruce, SIN BORRAR ont_id
        df_zabbix['CONTRATO'] = df_zabbix['ont_id'].astype(str).str.strip() #Crear llave cruce
        
        #Opcional: Limpiar ont_id original también
        df_zabbix['ont_id'] = df_zabbix['ont_id'].astype(str).str.strip() #Normalizar columna original

        df_zabbix = df_zabbix.drop_duplicates(subset=['ont_id'], keep='last') #Eliminar duplicados
        
        print(f"   Registros Zabbix Únicos: {len(df_zabbix)}") #Imprimir únicos zabbix

        print("   Cruzando información...") #Imprimir estado cruce
        #Left Join: Mantener servicios, pegar info zabbix
        df_enriched = pd.merge(df_silver, df_zabbix, on='CONTRATO', how='left') #Ejecutar fusión dataframes

        print("   Estandarizando tipos...") #Imprimir estado limpieza
        #Convertir objetos a string para evitar conflictos en Parquet
        for col in df_enriched.columns: #Recorrer columnas
            if df_enriched[col].dtype == 'object': #Verificar tipo objeto
                df_enriched[col] = df_enriched[col].astype(str) #Convertir a string

        print(f"   Guardando en {RUTA_SILVER_RX_AVG}...") #Imprimir estado guardado
        
        #Generar Parquet
        df_enriched.to_parquet(FILE_OUTPUT_PARQUET, index=False) #Exportar archivo parquet
        
        #Generar CSV (Para comparación visual)
        df_enriched.to_csv(FILE_OUTPUT_CSV, index=False, encoding='utf-8-sig') #Exportar archivo csv

        print(f"[EXITO] Parquet generado: {FILE_OUTPUT_PARQUET}") #Imprimir éxito parquet
        print(f"[EXITO] CSV generado: {FILE_OUTPUT_CSV}") #Imprimir éxito csv
        print(f"Total Registros: {len(df_enriched)}") #Imprimir total registros
        print(f"Columnas: {list(df_enriched.columns)}") #Imprimir lista columnas
        
        print("Muestra (Top 5):") #Imprimir cabecera muestra
        #Mostrar columnas clave, incluyendo la original ont_id
        print(df_enriched.head()) #Imprimir registros muestra
        
    except FileNotFoundError as e: #Capturar error archivo no encontrado
        print(f"[ERROR] No se pudo encontrar el archivo Master Silver base: {e}") #Imprimir mensaje error
    except Exception as e: #Capturar otros errores
        print(f"[ERROR] Ha ocurrido un error inesperado durante el cruce: {e}") #Imprimir mensaje error

if __name__ == "__main__": #Verificar ejecución directa
    unir_silver_zabbix() #Ejecutar función principal

W:\RUBEN\python\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
W:\RUBEN\python\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Iniciando Enriquecimiento Silver con Zabbix para el periodo 2602...
   Cargando Master Silver...
   Cargando Zabbix (ont_id + rx_avg)...
   Registros Zabbix Únicos: 10513
   Cruzando información...
   Estandarizando tipos...
   Guardando en datalake/silver/rx_avg...
[EXITO] Parquet generado: datalake/silver/rx_avg/master_rx+services_silver_2602.parquet
[EXITO] CSV generado: datalake/silver/rx_avg/master_rx+services_silver_2602.csv
Total Registros: 424
Columnas: ['CONTRATO', 'FECHA_PEDIDO', 'MOTIVO_PEDIDO', 'PAQUETE_x', 'DETALLE_PEDIDO1', 'DETALLE_PEDIDO2', 'FECHA_CUMPLIMIENTO', 'COD_TECNICO', 'ESTATUS_x', 'ZONA', 'MODEM(S)_x', 'ESTATUS_y', 'PAQUETE_y', 'FECHA_INSTALACION', 'MODEM(S)_y', 'DIAS_ANTIGUEDAD', 'DIAS_ATENCION', 'ont_id', 'rx_avg']
Muestra (Top 5):
  CONTRATO FECHA_PEDIDO                          MOTIVO_PEDIDO  \
0    42295   2026-02-01                          EQUIPO DA?ADO   
1    45662   2026-02-01            NO NAVEGA. LEDS FIJOS/RUIDO   
2    38705   2026-02-01          